# BreakHis project: Colab quickstart

Run the cells in order. Set **Runtime > Change runtime type > GPU** first.

## 1. Get the code
Replace `YOUR-GITHUB-USERNAME` with the account that owns the repo.

In [ ]:
REPO = "https://github.com/YOUR-GITHUB-USERNAME/breakhis-project.git"
!git clone -q $REPO project 2>/dev/null || (cd project && git pull -q)
%cd project
!pip install -q -r requirements.txt

## 2. Download BreakHis
Kaggle may ask you to log in once with your username and API token from kaggle.com > Settings > API.

In [ ]:
import kagglehub
HANDLE = "kurshidbasheer/breakhisbreast-cancer-histopathological-images"
try:
    DATA_DIR = kagglehub.dataset_download(HANDLE)
except Exception:
    kagglehub.login()
    DATA_DIR = kagglehub.dataset_download(HANDLE)
print(DATA_DIR)

## 3. Check the data and the patient split

In [ ]:
from breakhis.data import build_index, make_split, find_official_folds
df = build_index(DATA_DIR)
df['split'] = make_split(df, 'patient', seed=0).to_numpy()
print(len(df), 'images,', df.patient.nunique(), 'patients')
print('Official Folds.csv found:', find_official_folds(DATA_DIR) is not None)
df.groupby('split').agg(images=('path', 'count'), patients=('patient', 'nunique'), malignant_share=('label', 'mean'))

## 4. Quick sanity run
Two epochs on 400 images. It should finish in a few minutes and print test metrics.

In [ ]:
!python train.py --data-dir "$DATA_DIR" --model mobilenet_v2 --epochs 2 --limit 400 --run-name debug_quick --out-dir results_debug

## 5. One real experiment
RQ1 baseline: MobileNetV2 on the patient split. Swap the flags to run any other experiment from the README.

In [ ]:
!python train.py --data-dir "$DATA_DIR" --model mobilenet_v2 --epochs 15 --seed 0 --run-name e1_mobilenet_s0

## 6. Everything
All experiments for three seeds. This takes a while, so you may want to run it in pieces.

In [ ]:
# !bash scripts/run_experiments.sh "$DATA_DIR" 15

## 7. Results table

In [ ]:
!python collect_results.py

## 8. Save your results to the repo
Checkpoints are too big for git, so only the small JSON results get committed.

In [ ]:
# !git add results/*.json results/summary.*
# !git commit -m "Add results"
# !git push